# TICKIT Sales Investigation & XGBoost Forecast

This notebook analyzes the TICKIT sample dataset and builds an XGBoost model to forecast daily ticket sales revenue.

**Tables used:**
- `SAMPLES.TICKIT.SALES` — transaction-level sales records
- `SAMPLES.TICKIT.DATE` — date dimension with calendar metadata
- `SAMPLES.TICKIT.EVENT` — event details
- `SAMPLES.TICKIT.CATEGORY` — event categories

**Workflow:**
1. Exploratory data analysis
2. Aggregate daily sales
3. Feature engineering (lag features, rolling averages, calendar features)
4. Train/test split + XGBoost model
5. Evaluate and forecast

## 1. Exploratory Data Analysis

In [ ]:
%%sql -r sales_overview
SELECT
    MIN(SALETIME)::DATE AS first_sale,
    MAX(SALETIME)::DATE AS last_sale,
    COUNT(*)            AS total_transactions,
    SUM(QTYSOLD)        AS total_tickets_sold,
    ROUND(SUM(PRICEPAID), 2) AS total_revenue,
    ROUND(AVG(PRICEPAID), 2) AS avg_order_value
FROM SAMPLES.TICKIT.SALES

In [ ]:
%notebook_exit

In [ ]:
%%sql -r sales_by_category
SELECT
    c.CATGROUP,
    c.CATNAME,
    COUNT(s.SALESID)         AS num_transactions,
    SUM(s.QTYSOLD)           AS tickets_sold,
    ROUND(SUM(s.PRICEPAID), 2) AS total_revenue,
    ROUND(AVG(s.PRICEPAID), 2) AS avg_price
FROM SAMPLES.TICKIT.SALES s
JOIN SAMPLES.TICKIT.EVENT e ON s.EVENTID = e.EVENTID
JOIN SAMPLES.TICKIT.CATEGORY c ON e.CATID = c.CATID
GROUP BY 1, 2
ORDER BY total_revenue DESC

In [ ]:
%%sql -r monthly_sales
SELECT
    d.MONTH,
    d.QTR,
    COUNT(s.SALESID)            AS num_transactions,
    SUM(s.QTYSOLD)              AS tickets_sold,
    ROUND(SUM(s.PRICEPAID), 2)  AS total_revenue
FROM SAMPLES.TICKIT.SALES s
JOIN SAMPLES.TICKIT.DATE d ON s.DATEID = d.DATEID
GROUP BY 1, 2
ORDER BY
    CASE d.MONTH
        WHEN 'JAN' THEN 1 WHEN 'FEB' THEN 2 WHEN 'MAR' THEN 3
        WHEN 'APR' THEN 4 WHEN 'MAY' THEN 5 WHEN 'JUN' THEN 6
        WHEN 'JUL' THEN 7 WHEN 'AUG' THEN 8 WHEN 'SEP' THEN 9
        WHEN 'OCT' THEN 10 WHEN 'NOV' THEN 11 WHEN 'DEC' THEN 12
    END

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenue by category
cat_df = sales_by_category.copy()
cat_df = cat_df.sort_values('TOTAL_REVENUE', ascending=True)
axes[0].barh(cat_df['CATNAME'], cat_df['TOTAL_REVENUE'] / 1e6, color='steelblue')
axes[0].set_xlabel('Total Revenue ($M)')
axes[0].set_title('Revenue by Category')

# Monthly revenue
month_order = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
mon_df = monthly_sales.copy()
mon_df['MONTH'] = pd.Categorical(mon_df['MONTH'], categories=month_order, ordered=True)
mon_df = mon_df.sort_values('MONTH')
axes[1].bar(mon_df['MONTH'], mon_df['TOTAL_REVENUE'] / 1e6, color='darkorange')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Total Revenue ($M)')
axes[1].set_title('Monthly Revenue')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 2. Build Daily Sales Time Series

In [ ]:
%%sql -r daily_sales
SELECT
    d.CALDATE                     AS sale_date,
    d.YEAR,
    d.WEEK,
    d.MONTH,
    d.QTR,
    d.DAY,
    d.HOLIDAY,
    COUNT(s.SALESID)              AS num_transactions,
    SUM(s.QTYSOLD)                AS tickets_sold,
    ROUND(SUM(s.PRICEPAID), 2)    AS daily_revenue
FROM SAMPLES.TICKIT.DATE d
LEFT JOIN SAMPLES.TICKIT.SALES s ON d.DATEID = s.DATEID
GROUP BY 1, 2, 3, 4, 5, 6, 7
ORDER BY 1

In [ ]:
import pandas as pd

df = daily_sales.copy()
df['SALE_DATE'] = pd.to_datetime(df['SALE_DATE'])
df = df.sort_values('SALE_DATE').reset_index(drop=True)

# Fill NaN revenue (days with no sales) with 0
df['DAILY_REVENUE'] = df['DAILY_REVENUE'].fillna(0)
df['TICKETS_SOLD'] = df['TICKETS_SOLD'].fillna(0)
df['NUM_TRANSACTIONS'] = df['NUM_TRANSACTIONS'].fillna(0)

print(df.shape)
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['SALE_DATE'], df['DAILY_REVENUE'], linewidth=0.8, color='steelblue', label='Daily Revenue')
ax.fill_between(df['SALE_DATE'], df['DAILY_REVENUE'], alpha=0.2, color='steelblue')
ax.set_title('Daily Ticket Sales Revenue')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
df_feat = df.copy()

# --- Calendar features ---
df_feat['day_of_week']  = df_feat['SALE_DATE'].dt.dayofweek      # 0=Mon … 6=Sun
df_feat['day_of_month'] = df_feat['SALE_DATE'].dt.day
df_feat['week_of_year'] = df_feat['SALE_DATE'].dt.isocalendar().week.astype(int)
df_feat['month_num']    = df_feat['SALE_DATE'].dt.month
df_feat['is_weekend']   = (df_feat['day_of_week'] >= 5).astype(int)
df_feat['is_holiday']   = (df_feat['HOLIDAY'] == 'T').astype(int)
df_feat['quarter']      = df_feat['SALE_DATE'].dt.quarter

# --- Lag features (previous days' revenue) ---
for lag in [1, 2, 3, 7, 14]:
    df_feat[f'lag_{lag}'] = df_feat['DAILY_REVENUE'].shift(lag)

# --- Rolling window statistics ---
for window in [7, 14, 30]:
    df_feat[f'rolling_mean_{window}'] = df_feat['DAILY_REVENUE'].shift(1).rolling(window).mean()
    df_feat[f'rolling_std_{window}']  = df_feat['DAILY_REVENUE'].shift(1).rolling(window).std()

# Drop rows with NaN (caused by lags / rolling windows)
df_feat = df_feat.dropna().reset_index(drop=True)

print(f'Feature matrix shape: {df_feat.shape}')
df_feat[['SALE_DATE','DAILY_REVENUE','lag_1','lag_7','rolling_mean_7']].head()

## 4. Train / Test Split

In [ ]:
FEATURE_COLS = [
    'day_of_week', 'day_of_month', 'week_of_year', 'month_num',
    'is_weekend', 'is_holiday', 'quarter',
    'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14',
    'rolling_mean_7', 'rolling_std_7',
    'rolling_mean_14', 'rolling_std_14',
    'rolling_mean_30', 'rolling_std_30',
]
TARGET = 'DAILY_REVENUE'

# Chronological 80/20 split
split_idx = int(len(df_feat) * 0.8)
train = df_feat.iloc[:split_idx]
test  = df_feat.iloc[split_idx:]

X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET]

print(f'Train: {len(train)} rows  |  Test: {len(test)} rows')
print(f'Train ends: {train["SALE_DATE"].max().date()}  |  Test starts: {test["SALE_DATE"].min().date()}')

## 5. Train XGBoost Model

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

model = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Use symmetric MAPE to avoid division by near-zero actuals
smape = np.mean(2 * np.abs(y_test - y_pred) / (np.abs(y_test) + np.abs(y_pred) + 1e-6)) * 100

print(f'MAE   : ${mae:,.2f}')
print(f'RMSE  : ${rmse:,.2f}')
print(f'sMAPE : {smape:.1f}%  (symmetric MAPE, robust to near-zero days)')

In [ ]:
# --- Actual vs Predicted on test set ---
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(test['SALE_DATE'].values, y_test.values, label='Actual', color='steelblue', linewidth=1.2)
ax.plot(test['SALE_DATE'].values, y_pred, label='XGBoost Predicted', color='darkorange', linewidth=1.2, linestyle='--')
ax.set_title('Actual vs Predicted — Test Period')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

print(f'MAE={mae:,.0f}  |  RMSE={rmse:,.0f}  |  sMAPE={smape:.1f}%')

In [ ]:
# --- Feature importance ---
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(importances.index, importances.values, color='teal')
ax.set_title('XGBoost Feature Importances')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 6. Forecast Future 30 Days

Since the dataset covers a single calendar year, we simulate a 30-day forecast by iteratively predicting one day at a time using each previous prediction as the next lag.

In [ ]:
from datetime import timedelta

FORECAST_DAYS = 30

# Seed the rolling history with the last known actuals
history = df_feat[['SALE_DATE', 'DAILY_REVENUE']].copy()

last_date = history['SALE_DATE'].max()
forecast_rows = []

for i in range(1, FORECAST_DAYS + 1):
    future_date = last_date + timedelta(days=i)
    rev_series  = history['DAILY_REVENUE']

    row = {
        'SALE_DATE':    future_date,
        'day_of_week':  future_date.dayofweek,
        'day_of_month': future_date.day,
        'week_of_year': future_date.isocalendar()[1],
        'month_num':    future_date.month,
        'is_weekend':   int(future_date.dayofweek >= 5),
        'is_holiday':   0,
        'quarter':      (future_date.month - 1) // 3 + 1,
        'lag_1':  rev_series.iloc[-1],
        'lag_2':  rev_series.iloc[-2],
        'lag_3':  rev_series.iloc[-3],
        'lag_7':  rev_series.iloc[-7],
        'lag_14': rev_series.iloc[-14],
        'rolling_mean_7':  rev_series.iloc[-7:].mean(),
        'rolling_std_7':   rev_series.iloc[-7:].std(),
        'rolling_mean_14': rev_series.iloc[-14:].mean(),
        'rolling_std_14':  rev_series.iloc[-14:].std(),
        'rolling_mean_30': rev_series.iloc[-30:].mean(),
        'rolling_std_30':  rev_series.iloc[-30:].std(),
    }

    X_future = pd.DataFrame([row])[FEATURE_COLS]
    pred_rev  = float(model.predict(X_future)[0])
    pred_rev  = max(pred_rev, 0)

    forecast_rows.append({'SALE_DATE': future_date, 'FORECASTED_REVENUE': pred_rev})

    # Append prediction to history for next iteration's lags
    new_row = pd.DataFrame([{'SALE_DATE': future_date, 'DAILY_REVENUE': pred_rev}])
    history = pd.concat([history, new_row], ignore_index=True)

forecast_df = pd.DataFrame(forecast_rows)
forecast_df

In [ ]:
# --- Plot historical + forecast ---
historical_plot = df[['SALE_DATE', 'DAILY_REVENUE']].copy()
historical_plot = historical_plot[historical_plot['DAILY_REVENUE'] > 0]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(historical_plot['SALE_DATE'], historical_plot['DAILY_REVENUE'],
        color='steelblue', linewidth=0.9, label='Historical')
ax.plot(forecast_df['SALE_DATE'], forecast_df['FORECASTED_REVENUE'],
        color='crimson', linewidth=1.5, linestyle='--', marker='o', markersize=4,
        label='30-Day Forecast')
ax.axvline(historical_plot['SALE_DATE'].max(), color='gray', linestyle=':', linewidth=1)
ax.set_title('TICKIT Daily Revenue — Historical + 30-Day XGBoost Forecast')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nForecasted total 30-day revenue: ${forecast_df['FORECASTED_REVENUE'].sum():,.2f}")
print(f"Average daily forecasted revenue: ${forecast_df['FORECASTED_REVENUE'].mean():,.2f}")